# Week 12: Building a Group Project, Step by Step

**COMP 1110: Pokemon Battle Helper**

A small worked example: README → `project.py` → `test_project.py`

**Repository:** [github.com/quan3010/comp1110_group_00](https://github.com/quan3010/comp1110_group_00)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TRU-CS/COMP_1110/blob/main/COMP_1110/notebooks/week12_pokemon_project_walkthrough.ipynb)

*Run this notebook yourself: press the power button at the top of the page to
execute cells right here, or click the badge to open an editable copy in Google
Colab (a Google account is required).*


## The project at a glance

![Overview: user input, project.py with STRONG and MOVES and three functions, output, and three passing tests](../img/week12_pokemon_overview.png)

## The plan

1. **README first.** Write the idea in plain English.
2. **`project.py`.** Data first, then one function at a time, then `main()`.
3. **`if __name__ == "__main__":`**
4. **`test_project.py`.** One test per function.

> Think before you type. If you can't explain it in English, you can't code it.

# Step 1: The README (plain English)

## What is the project?

> **Pokemon Battle Helper.** You pick a Pokemon (Charmander, Squirtle or
> Bulbasaur) and type the opponent's type. The program tells you which move
> to use and how much damage it does.

**The rule:** fire beats grass, water beats fire, grass beats water.

No code yet. Just the idea.

## The three functions, in plain English

| Function | What it does |
|---|---|
| `get_effectiveness` | Given an attack type and a defending type, say if the attack is **strong (×2)**, **weak (×0.5)** or **normal (×1)**. |
| `calculate_damage` | Given a move's power and that multiplier, work out the damage. Never less than 1. |
| `best_move` | Given my Pokemon and the opponent's type, pick my Pokemon's move (Ember, Water Gun or Vine Whip), unless it is weak; then use Tackle. |

Plus **`main()`**: asks the questions and prints the answer.

One function per team member, so everyone owns a piece.

## The README, version 1

```markdown
# Pokemon Battle Helper

#### Repository: https://github.com/quan3010/comp1110_group_00

## Description
Pick a Pokemon and the opponent's type, and the program suggests
a move and its damage. Fire beats grass, water beats fire,
grass beats water.

## Functions
- get_effectiveness: is my attack strong, weak or normal?
- calculate_damage: how much damage does the move do?
- best_move: which move should I use?
- main: asks the user and prints the advice.
```

Commit this **before** writing code. You will add to it as you build.

# Step 2: `project.py`

## Start with a docstring and the data

What does the program need to *know*?

1. Which **type** beats which (fire, water, grass).
2. Which **move** each Pokemon uses (Ember, Water Gun, Vine Whip), and that move's type.

Both are "look up a key, get a value", so both are **dictionaries**.

> **Types** are lowercase words like `"fire"`. **Moves** are names like `"Ember"`.
> Keeping them apart makes the code easier to read.

## `STRONG`: who beats whom

Key = a type, value = the type it beats.

In [1]:
"""Pokemon Battle Helper: suggest the best move against an opponent."""

STRONG = {"fire": "grass", "water": "fire", "grass": "water"}

print(STRONG["water"])       # what does water beat?

fire


One dictionary answers both questions:

- Is **water** strong against **fire**? `STRONG["water"] == "fire"`, so yes.
- Is **fire** weak against **water**? `STRONG["water"] == "fire"`, so yes, the other way round.

## `MOVES`: each Pokemon's move and its type

Key = Pokemon name, value = a **tuple** `(move, type)` *(Week 11)*.
The move's name and its type stay together, so we need only one dictionary.

In [2]:
MOVES = {"charmander": ("Ember", "fire"),
         "squirtle": ("Water Gun", "water"),
         "bulbasaur": ("Vine Whip", "grass")}

move, move_type = MOVES["squirtle"]   # unpack the tuple
print(move, "is a", move_type, "move")
print("pikachu" in MOVES)    # handy for checking user input later

Water Gun is a water move
False


## Function 1: `get_effectiveness`

**English:** If my type beats theirs, ×2. If theirs beats mine, ×0.5. Otherwise ×1.

In [3]:
def get_effectiveness(attack_type, defend_type):
    """Return 2 if attack_type beats defend_type, 0.5 if it loses, else 1."""
    if STRONG.get(attack_type) == defend_type:
        return 2
    if STRONG.get(defend_type) == attack_type:
        return 0.5
    return 1

Why `.get()` and not `STRONG[attack_type]`? Because `"normal"` is not a key.
`.get()` gives `None` instead of crashing.

## Try it before moving on

In [4]:
print(get_effectiveness("water", "fire"))   # strong
print(get_effectiveness("fire", "water"))   # weak
print(get_effectiveness("fire", "fire"))    # normal
print(get_effectiveness("normal", "fire"))  # not in STRONG, so normal

2
0.5
1
1


## Function 2: `calculate_damage`

**English:** damage = power × multiplier, as a whole number, and at least 1.

In [5]:
def calculate_damage(power, multiplier):
    """Return the whole-number damage of a move, at least 1."""
    return max(1, int(power * multiplier))

In [6]:
print(calculate_damage(40, 2))     # 80
print(calculate_damage(40, 0.5))   # 20
print(calculate_damage(1, 0.5))    # int(0.5) is 0, but max makes it 1

80
20
1


## Function 3: `best_move`

**English:** Look up my Pokemon's move and its type. If the type is not weak
against the opponent, use that move. Otherwise, use **Tackle**, a backup move
of type `"normal"` (never strong or weak). Return the move *and* its type.

In [7]:
def best_move(pokemon, defend_type):
    """Return (move, type): the pokemon's own move, or Tackle if it is weak."""
    move, move_type = MOVES[pokemon]
    if get_effectiveness(move_type, defend_type) >= 1:
        return move, move_type
    return "Tackle", "normal"

Notice `best_move` **reuses** `get_effectiveness`. Functions built on functions.

## Try it

In [8]:
print(best_move("squirtle", "fire"))     # Water Gun: water beats fire
print(best_move("bulbasaur", "grass"))   # Vine Whip: grass vs grass is normal
print(best_move("charmander", "water"))  # Ember is weak, so Tackle

('Water Gun', 'water')
('Vine Whip', 'grass')
('Tackle', 'normal')


## `main()`: the only part that talks to the user

**English:** Ask for a Pokemon until it is valid. Ask for a type until it is
valid. Pick the move, work out the damage, print it.

In [9]:
def main():
    """Ask for a pokemon and opponent type, then suggest a move."""
    pokemon = input("Your pokemon (charmander/squirtle/bulbasaur): ").lower()
    while pokemon not in MOVES:
        pokemon = input("Try again: ").lower()
    enemy = input("Opponent type (fire/water/grass): ").lower()
    while enemy not in STRONG:
        enemy = input("Try again: ").lower()
    move, move_type = best_move(pokemon, enemy)
    power = calculate_damage(40, get_effectiveness(move_type, enemy))
    print(f"Use {move} for {power} damage!")

## Why keep `input()` and `print()` in `main` only?

- The three helper functions take **arguments** and **return** values.
- `pytest` can't type into `input()`, but it *can* call
  `best_move("squirtle", "fire")` and check the answer.
- `while ... not in MOVES` means bad input is asked again, not a crash.

Run `main()` in the next cell to try it.

In [10]:
# main()   # remove the # to try it in the notebook

## Step 3: `if __name__ == "__main__":`

The last two lines of `project.py`:

```python
if __name__ == "__main__":
    main()
```

| How the file is used | `__name__` is | `main()` runs? |
|---|---|---|
| `python3 project.py` | `"__main__"` | **yes** |
| `from project import best_move` (pytest does this) | `"project"` | **no** |

Without it, `pytest` would import the file, hit `input()`, and hang.

## The whole `project.py` (45 lines)

```python
"""Pokemon Battle Helper: suggest the best move against an opponent."""

STRONG = {"fire": "grass", "water": "fire", "grass": "water"}
MOVES = {"charmander": ("Ember", "fire"),
         "squirtle": ("Water Gun", "water"),
         "bulbasaur": ("Vine Whip", "grass")}


def get_effectiveness(attack_type, defend_type): ...


def calculate_damage(power, multiplier): ...


def best_move(pokemon, defend_type): ...


def main(): ...


if __name__ == "__main__":
    main()
```

Two blank lines between functions (PEP 8). Run it: `python3 project.py`

# Step 4: `test_project.py`

## The test file

- Lives next to `project.py`.
- Imports the three functions (this is where `__name__` matters).
- One test per function, named `test_` + the function name.
- Each test is a few `assert`s. If any is false, the test fails.

```python
"""Tests for project.py. Run with: pytest test_project.py"""

from project import get_effectiveness, calculate_damage, best_move
```

## `test_get_effectiveness`

Test **each branch**: strong, weak, normal.

In [11]:
def test_get_effectiveness():
    """Strong, weak and neutral matchups."""
    assert get_effectiveness("water", "fire") == 2
    assert get_effectiveness("fire", "water") == 0.5
    assert get_effectiveness("fire", "fire") == 1


test_get_effectiveness()   # pytest calls this for you; here we call it
print("passed")

passed


## `test_calculate_damage`

Test the normal cases **and the edge case** (the minimum of 1).

In [12]:
def test_calculate_damage():
    """Damage is power times multiplier, never below 1."""
    assert calculate_damage(40, 2) == 80
    assert calculate_damage(40, 0.5) == 20
    assert calculate_damage(1, 0.5) == 1


test_calculate_damage()
print("passed")

passed


## `test_best_move`

Test **both outcomes**: use your own move, or fall back to Tackle.

In [13]:
def test_best_move():
    """Use your own move unless it is weak, then Tackle."""
    assert best_move("squirtle", "fire") == ("Water Gun", "water")
    assert best_move("bulbasaur", "grass") == ("Vine Whip", "grass")
    assert best_move("charmander", "water") == ("Tackle", "normal")


test_best_move()
print("passed")

passed


## What a failing test looks like

Suppose someone "fixes" `calculate_damage` and forgets the `max`:

In [14]:
def calculate_damage(power, multiplier):
    """Broken on purpose: no minimum of 1."""
    return int(power * multiplier)


try:
    test_calculate_damage()
except AssertionError:
    print("test_calculate_damage FAILED: the test caught the bug")

test_calculate_damage FAILED: the test caught the bug


## Running the tests for real

From the project folder, in the terminal:

```
$ pytest test_project.py
...                                                     [100%]
3 passed in 0.01s
```

Each `.` is one passing test. An `F` is a failure, and pytest shows the
`assert` that went wrong.

## Recap

1. **README:** the idea and the three functions in plain English.
2. **Data:** `STRONG` and `MOVES` dictionaries.
3. **Functions:** one at a time, try each before moving on.
4. **`main()`:** the only place with `input()` / `print()`.
5. **`if __name__ == "__main__":`** so pytest can import safely.
6. **Tests:** one `test_<name>` per function, cover each branch.

Your project: the same shape, with **three functions per member**.

Full code: [github.com/quan3010/comp1110_group_00](https://github.com/quan3010/comp1110_group_00)